# Glass brain plotting
Visualises functional connectivity for a single participant as a glass brain. Execute cells top to bottom. Only cells marked **[CONFIGURE]** require changes.

> Run `scripts/run_scripts.ipynb` before running this notebook.

## 1. Environment Setup **[OPTIONAL]**
Mounts Google Drive and installs dependencies when running on Colab. Skip if running locally.

> **NOTE: requires moving `PROJECT` folder to Google Drive.**

In [ ]:
# COLAB
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd '/content/drive/My Drive/PROJECT'

    !pip install -q numpy pandas scikit-learn nilearn seaborn matplotlib

    sys.path.insert(0, './product/src')

## 2. Imports

In [ ]:
from nilearn import image, plotting
from nilearn.plotting import find_parcellation_cut_coords, plot_connectome
from nilearn.connectome import vec_to_sym_matrix
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_selection import SelectKBest, f_classif
from data_io.save_load_dataset import load_dataset
from utils.paths import get_project_root

## 3. Configuration **[CONFIGURE]**

| Variable | Description |
|---|---|
| `PARTICIPANT_IDX` | Row index of the participant to visualise |
| `N_FEATURES` | Number of top ANOVA features to select |
| `EDGE_THRESHOLD` | Minimum absolute FC weight required to draw an edge |
| `DISPLAY_MODE` | Projection view: `'ortho'`, `'x'`, `'y'`, or `'z'` |

In [ ]:
# ---- CHANGE THESE ------------------------------------------------
PARTICIPANT_IDX = 0       # which participant to visualise
N_FEATURES      = 435     # number of top ANOVA features to select
EDGE_THRESHOLD  = 0.5     # only draw edges where |FC weight| exceeds this
DISPLAY_MODE    = 'ortho' # 'ortho', 'x', 'y', or 'z'
# ------------------------------------------------------------------

## 4. Load Data

Loads the atlas, AAL map, region coordinates, and the ABIDE dataset. The dataset prefix is fixed to `'1100'` (full dataset). Update the prefix argument in `load_dataset` to use a subset.

In [ ]:
data       = get_project_root() / 'product' / 'data'
atlas_path = str(data / 'external' / 'cc200_roi_atlas.nii.gz')
aal_map    = pd.read_csv(data / 'external' / 'AAL_map.csv')
coords_df  = pd.read_csv(data / 'processed' / 'coords.csv')

atlas_img = image.load_img(atlas_path)
coords    = find_parcellation_cut_coords(atlas_path)
print(f"Extracted {len(coords)} region coordinates.")

prefix = "1100"

X, X_raw, y, metadata, feature_labels = load_dataset(prefix, verbose=False)
print(f"Participant:\n{metadata.iloc[PARTICIPANT_IDX]}")

## 5. Feature Selection and Matrix Reconstruction

Selects the top `N_FEATURES` features by ANOVA F-value, then reconstructs a full 200x200 symmetric connectivity matrix for the chosen participant. Fisher z-scores are converted back to Pearson r via `tanh`, then normalised to [-1, 1].

In [ ]:
N_ROIS           = 200
selector         = SelectKBest(score_func=f_classif, k=N_FEATURES)
X_selected       = selector.fit_transform(X, y)
selected_indices = selector.get_support(indices=True)
X_pearson        = np.tanh(X_selected)

total_features  = N_ROIS * (N_ROIS - 1) // 2
X_full_vector   = np.zeros(total_features)
X_full_vector[selected_indices] = X_pearson[PARTICIPANT_IDX]

matrix    = np.zeros((N_ROIS, N_ROIS))
iu        = np.triu_indices(N_ROIS, k=1)
matrix[iu] = X_full_vector
matrix    = matrix + matrix.T
np.fill_diagonal(matrix, 0)

# Normalise to [-1, 1]
pos_mask = matrix > 0
neg_mask = matrix < 0
if np.any(neg_mask):
    matrix[neg_mask] = matrix[neg_mask] / np.abs(matrix[neg_mask].min())
if np.any(pos_mask):
    matrix[pos_mask] = matrix[pos_mask] / matrix[pos_mask].max()

print(matrix.shape)

## 6. Colour Setup

Assigns each node a colour based on its brain network. Builds a diverging colour map for edges, running from blue (negative correlation) through white to red (positive correlation).

In [ ]:
label_to_network = dict(zip(aal_map['NOTATION'], aal_map['NETWORK']))
unique_networks  = aal_map['NETWORK'].unique()
palette_colours  = sns.color_palette("Set3", len(unique_networks))
network_palette  = {net: palette_colours[i] for i, net in enumerate(unique_networks)}

node_colors = [
    network_palette.get(label_to_network.get(label), (0.5, 0.5, 0.5))
    for label in coords_df['label']
]

slate_blue  = "#4573C4"
salmon_red  = "#E35959"
custom_cmap = mcolors.LinearSegmentedColormap.from_list(
    "slate_salmon_diverging", [slate_blue, "#FFFFFF", salmon_red]
)

## 7. Node Plot

Plots all 200 CC200 ROI nodes on a glass brain, coloured by network. No edges are shown. Uncomment the `savefig` line to save as PNG.

In [ ]:
fig = plt.figure(figsize=(8, 5), dpi=300)

custom_nodes_cmap = ListedColormap(node_colors)
node_indices      = np.arange(len(node_colors))

display = plotting.plot_markers(
    node_values=node_indices,
    node_coords=coords,
    node_cmap=custom_nodes_cmap,
    node_size=100,
    display_mode=DISPLAY_MODE,
    black_bg=False,
    alpha=1,
    colorbar=False,
    figure=fig,
    node_kwargs={'edgecolor': 'black', 'linewidth': 0.7},
)

legend_handles = [
    mpatches.Patch(facecolor=color, label=net, edgecolor='black', linewidth=1)
    for net, color in network_palette.items()
]

main_ax = display.axes['y'].ax
main_ax.legend(
    handles=legend_handles,
    title="Brain Networks",
    labelspacing=1.2,
    loc='upper center',
    bbox_to_anchor=(1.7, -0.1),
    ncol=4,
    frameon=False,
    fontsize='medium',
    title_fontsize='medium',
    columnspacing=1.0,
)

plt.subplots_adjust(bottom=0.25)
# plt.savefig("brain_networks_nodes_300dpi.png", dpi=300, bbox_inches='tight')
plt.show()

## 8. Connectome Plot

Plots edges between ROIs where the absolute connectivity weight exceeds `EDGE_THRESHOLD`. Edge colour encodes the direction and strength of connectivity using the diverging colour map. Uncomment the `savefig` line to save as PNG.

In [ ]:
thresholded = matrix.copy()
thresholded[np.abs(thresholded) <= EDGE_THRESHOLD] = 0

fig_conn = plt.figure(figsize=(12, 6), dpi=300)

display = plotting.plot_connectome(
    adjacency_matrix=thresholded,
    node_coords=coords,
    node_color=node_colors,
    node_size=100,
    edge_cmap=custom_cmap,
    edge_vmin=-1,
    edge_vmax=1,
    edge_threshold=0,
    display_mode=DISPLAY_MODE,
    black_bg=False,
    colorbar=True,
    figure=fig_conn,
    title=None,
    alpha=1.0,
    node_kwargs={'edgecolor': 'black', 'linewidth': 0.7, 'alpha': 1.0},
    edge_kwargs={'linewidth': 2, 'alpha': 1},
)

network_handles = [
    mpatches.Patch(facecolor=color, label=net, edgecolor='black', linewidth=0.8)
    for net, color in network_palette.items()
]

main_ax = display.axes['y'].ax
main_ax.legend(
    handles=network_handles,
    title="Brain Networks",
    title_fontsize='medium',
    fontsize='medium',
    loc='upper center',
    bbox_to_anchor=(1.7, -0.1),
    ncol=4,
    frameon=False,
    columnspacing=1.0,
    labelspacing=0.6,
)

plt.subplots_adjust(bottom=0.25)
# plt.savefig("brain_connectome_edges_300dpi.png", dpi=300, bbox_inches='tight')
plt.show()